# Drone Audio Classification â€” Kaggle Training

**Before running:**
- Settings > Accelerator > GPU T4 x2
- Settings > Internet > On
- Add dataset: `aranagnost/drone-audio-dataset`

## 1. Verify GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2. Clone the repo

In [ ]:
import os
import sys

REPO_DIR = '/kaggle/working/drone-audio-classification'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aranagnost/drone-audio-classification.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 3. Check dataset path

In [7]:
import os
print(os.listdir('/kaggle/input/datasets/aranagnost/drone-audio-dataset'))

['datasets']


In [9]:
DATASET_ROOT = '/kaggle/input/datasets/aranagnost/drone-audio-dataset/datasets/Drone_Audio_Dataset/audio'
print('Dataset root exists:', os.path.exists(DATASET_ROOT))
print('Subdirs:', os.listdir(DATASET_ROOT))

Dataset root exists: True
Subdirs: ['4_motors', '8_motors', '2_motors', 'not_a_drone', '6_motors']


## 4. Create artifacts directory

In [ ]:
os.makedirs('artifacts/checkpoints', exist_ok=True)
CACHE_DIR = '/kaggle/working/ast_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

## 5. Install transformers (for AST)

In [ ]:
!pip install -q transformers

## 6. Train AST Stage 2 (motor count: 2/4/6/8 motors)

**Changes vs v1:**
- **Frozen encoder**: only the classifier head (~3K params) is trained instead of all 86M. v1 overfit immediately (best epoch = 1, val loss rising from epoch 2); with ~5,700 training samples and 86M params the model memorised training data. Freezing eliminates this.
- **SpecAugment**: time masking (2×up to 40 frames) + frequency masking (2×up to 20 bins) applied during training only — baked into the dataset, no extra flag needed.
- **Stratified splits**: `data/splits_stage2/` regenerated so val and test have matching quality distributions within each motor class (previously val/test had very different q3 vs q4-5 ratios, causing the 0.67 val F1 → 0.31 test F1 gap).
- Checkpoint saved as `stage2_ast_v2.pt`

In [ ]:
os.environ['PYTHONPATH'] = REPO_DIR
!python training/train_ast.py --task stage2 --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --batch 16 --num_workers 4 --out artifacts/checkpoints/stage2_ast_v2.pt --freeze_encoder

## 7. Evaluate AST (Stage 2)

In [ ]:
!python training/train_ast.py --task stage2 --eval --dataset_root {DATASET_ROOT} --cache_dir {CACHE_DIR} --out artifacts/checkpoints/stage2_ast_v2.pt